# Reto 1 - Caracterización de variables aleatorias

## Caso de estudio

FitBalance Gym es una cadena ficticia de gimnasios que busca analizar el riesgo económico relacionado con la cancelación de membresías.

Frecuencia: indica el número de clientes que cancelan su membresía diariamente (clientes/día).

Severidad: representa el ingreso promedio que el gimnasio deja de recibir por cada cliente que cancela (COP/cliente).

Distribución de pérdidas: resulta de multiplicar la frecuencia por la severidad y permite estimar la pérdida económica diaria causada por las cancelaciones.

## 1. Librerías


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = "/content/drive/MyDrive/Integracion de datos/Reto_1_Clientes_Gimnasio.xlsx"

In [ ]:
df = pd.read_excel(path, sheet_name='Datos')
df.head()

Freq = pd.to_numeric(df.iloc[:,5], errors='coerce')
Sev = pd.to_numeric(df.iloc[:,6], errors='coerce')
LDA = Freq * Sev

plt.figure()
sns.histplot(LDA, color='blue', bins=10, kde=True)
plt.title('Distribución de pérdidas por cancelación de membresías')
plt.xlabel('Pérdidas')
plt.ylabel('Frecuencia')
plt.grid()
plt.show()

In [ ]:
df = pd.read_excel(path, sheet_name='Datos')
df.head()

Freq = pd.to_numeric(df.iloc[:,5], errors='coerce')
Sev = pd.to_numeric(df.iloc[:,6], errors='coerce')
LDA = Freq * Sev

plt.figure()
sns.histplot(LDA, color='blue', bins=10, kde=True)
plt.title('Distribución de pérdidas por cancelación de membresías')
plt.xlabel('Pérdidas')
plt.ylabel('Frecuencia')
plt.grid()
plt.show()

Se procede con la construcción del histograma de frecuencias

In [ ]:
NI = 10 # Número de intervalos
counts, Limits = np.histogram(LDA, bins=NI) # Agrupar los datos
LI = Limits[:-1] # Límite inferior
LS = Limits[1:] # Límite superior
fi = counts / sum(counts) # Porcentaje de datos por intervalo
MC = (LI + LS) / 2 # Marca de clase

histogram_data = pd.DataFrame({
    'LI': LI,
    'LS': LS,
    'ND': counts,
    'fi': fi,
    'MC': MC
})
display(histogram_data)

Medidas de tendencia central y dispersión

In [ ]:
np.set_printoptions(suppress=True)

# Media
U = np.sum(MC * fi)
print(f"La media de las pérdidas es: {U:.2f}")

# Varianza
varc = np.sum((MC - U)**2 * fi)
print(f"La varianza es: {varc:.2f}")

# Intervalo de variación de las pérdidas
sigma = np.sqrt(varc)
print(f"El límite inferior es: {U - 2*sigma:.2f}")
print(f"El límite superior es: {U + 2*sigma:.2f}")

# Coeficiente de asimetría
cas = np.sum(((MC - U)**3) * fi) / (sigma**3)
print(f"El coeficiente de asimetría es: {cas:.2f}")

# Coeficiente de curtosis
cur = np.sum(((MC - U)**4) * fi) / (sigma**4)
print(f"El coeficiente de curtosis es: {cur:.2f}")

## 5. Medidas solicitadas

In [ ]:
media = np.sum(MC * fi)
varianza = np.sum((MC - media)**2 * fi)
desviacion = np.sqrt(varianza)
asimetria = np.sum((MC - media)**3 * fi) / desviacion**3
curtosis = np.sum((MC - media)**4 * fi) / desviacion**4

resultados = pd.DataFrame({
    "Medida": ["Media", "Varianza", "Desviación estándar", "Asimetría", "Curtosis"],
    "Resultado": [media, varianza, desviacion, asimetria, curtosis]
})

resultados

In [ ]:
def MedidasDTC(Xi, NI=10):
    """Calcula las medidas pedidas por el reto usando el método de datos agrupados visto en clase."""
    Xi = pd.Series(Xi).dropna().to_numpy(dtype=float)

    counts, Limits = np.histogram(Xi, bins=NI)
    LI = Limits[:-1]
    LS = Limits[1:]
    fi = counts / counts.sum()
    MC = (LI + LS) / 2

    media = np.sum(MC * fi)
    varianza = np.sum((MC - media)**2 * fi)
    desviacion = np.sqrt(varianza)
    asimetria = np.sum((MC - media)**3 * fi) / desviacion**3
    curtosis = np.sum((MC - media)**4 * fi) / desviacion**4

    tabla_hist = pd.DataFrame({
        'Limite Inferior': LI,
        'Limite Superior': LS,
        'ND': counts,
        'fi': fi,
        'MC': MC
    })

    medidas = pd.Series({
        'Media': media,
        'Varianza': varianza,
        'Desviación estándar': desviacion,
        'Asimetría': asimetria,
        'Curtosis': curtosis
    })

    return tabla_hist, medidas


## 6. Reto 2 - Muestreo aleatorio siguiendo el procedimiento trabajado en clase

Para esta parte se mantiene la lógica del ejercicio visto en clase:

1. Se divide la LDA en cajas o intervalos.
2. Se calcula la proporción de datos que cae en cada caja.
3. Esa proporción se convierte en cantidad de datos a seleccionar.
4. Se utiliza `random.choices()` para construir una nueva muestra.
5. Se vuelven a calcular Media, Varianza, Asimetría y Curtosis.
6. Se comparan las medidas antes y después del muestreo.

Se realizan dos tamaños de muestra: 1.000 y 10.000 datos, de acuerdo con los niveles solicitados en el reto.


In [ ]:
import random

# Limpiar la distribución agregada de pérdidas
LDA_original = pd.Series(LDA).dropna().to_numpy(dtype=float)

print("Cantidad de datos originales:", len(LDA_original))


### 6.1 Construcción de las cajas


In [ ]:
# En el ejercicio de clase se trabaja con cajas.
# Para mantener esa lógica se utilizarán 5 intervalos.

NC = 5

counts_box, limits_box = np.histogram(LDA_original, bins=NC)

Boxes = []

for j in range(NC):
    if j == NC - 1:
        caja = LDA_original[
            (LDA_original >= limits_box[j]) &
            (LDA_original <= limits_box[j + 1])
        ]
    else:
        caja = LDA_original[
            (LDA_original >= limits_box[j]) &
            (LDA_original < limits_box[j + 1])
        ]

    Boxes.append(list(caja))

# Probabilidad o porcentaje de datos por caja
pm = counts_box / np.sum(counts_box)

tabla_cajas = pd.DataFrame({
    "Caja": np.arange(1, NC + 1),
    "Límite inferior": limits_box[:-1],
    "Límite superior": limits_box[1:],
    "Número de datos": counts_box,
    "Proporción": pm
})

display(tabla_cajas)


### 6.2 Medidas antes del muestreo


In [ ]:
hist_original, medidas_original = MedidasDTC(LDA_original, NI=10)

print("MEDIDAS ANTES DEL MUESTREO")
display(medidas_original.to_frame("Original"))


### 6.3 Primera muestra: 1.000 datos


In [ ]:
# Convertir las proporciones en cantidades para una muestra de 1.000 datos
pm_1000 = (np.array(pm) * 1000).astype(int)

# Ajustar por redondeo para obtener exactamente 1.000 datos
pm_1000[-1] = pm_1000[-1] + (1000 - np.sum(pm_1000))

print("Número de datos a sacar de cada caja:")
print(pm_1000)
print("Total:", np.sum(pm_1000))

# Construcción de la nueva LDA siguiendo el procedimiento de clase
LDA2 = []

random.seed(123)

for j in range(NC):
    if pm_1000[j] > 0 and len(Boxes[j]) > 0:
        LDA2.extend(random.choices(Boxes[j], k=pm_1000[j]))

LDA2 = np.array(LDA2)

print("Cantidad de datos de LDA2:", len(LDA2))


In [ ]:
# Medidas después del primer muestreo
hist_1000, medidas_1000 = MedidasDTC(LDA2, NI=10)

print("MEDIDAS DESPUÉS DEL MUESTREO - 1.000 DATOS")
display(medidas_1000.to_frame("Muestra 1.000"))


### 6.4 Segunda muestra: 10.000 datos


In [ ]:
# Convertir las proporciones en cantidades para una muestra de 10.000 datos
pm_10000 = (np.array(pm) * 10000).astype(int)

# Ajustar por redondeo para obtener exactamente 10.000 datos
pm_10000[-1] = pm_10000[-1] + (10000 - np.sum(pm_10000))

print("Número de datos a sacar de cada caja:")
print(pm_10000)
print("Total:", np.sum(pm_10000))

# Construcción de la segunda muestra
LDA3 = []

random.seed(456)

for j in range(NC):
    if pm_10000[j] > 0 and len(Boxes[j]) > 0:
        LDA3.extend(random.choices(Boxes[j], k=pm_10000[j]))

LDA3 = np.array(LDA3)

print("Cantidad de datos de LDA3:", len(LDA3))


In [ ]:
# Medidas después del segundo muestreo
hist_10000, medidas_10000 = MedidasDTC(LDA3, NI=10)

print("MEDIDAS DESPUÉS DEL MUESTREO - 10.000 DATOS")
display(medidas_10000.to_frame("Muestra 10.000"))


## 7. Comparación antes y después


In [ ]:
comparacion = pd.concat([
    medidas_original.rename("Original"),
    medidas_1000.rename("Muestra 1.000"),
    medidas_10000.rename("Muestra 10.000")
], axis=1)

display(comparacion)


## 8. Diferencia porcentual respecto a los datos originales


In [ ]:
comparacion_error = comparacion.copy()

comparacion_error["Diferencia % - 1.000"] = (
    (comparacion_error["Muestra 1.000"] - comparacion_error["Original"])
    / comparacion_error["Original"]
) * 100

comparacion_error["Diferencia % - 10.000"] = (
    (comparacion_error["Muestra 10.000"] - comparacion_error["Original"])
    / comparacion_error["Original"]
) * 100

display(comparacion_error)


## 9. Gráficas para apoyar el análisis


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(LDA_original, bins=10, kde=True)
plt.title("Distribución original de pérdidas")
plt.xlabel("Pérdida económica diaria")
plt.ylabel("Frecuencia")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(LDA2, bins=10, kde=True)
plt.title("Distribución de la muestra de 1.000 datos")
plt.xlabel("Pérdida económica diaria")
plt.ylabel("Frecuencia")
plt.grid()
plt.show()

plt.figure(figsize=(8,5))
sns.histplot(LDA3, bins=10, kde=True)
plt.title("Distribución de la muestra de 10.000 datos")
plt.xlabel("Pérdida económica diaria")
plt.ylabel("Frecuencia")
plt.grid()
plt.show()


## Análisis de resultados

Los resultados obtenidos permiten observar el efecto que tiene el tamaño de la muestra sobre la representación de la distribución original de pérdidas. Para esto se compararon los datos originales con dos muestras, una de 1.000 y otra de 10.000 observaciones, evaluando los cambios en la media, la varianza, la asimetría y la curtosis.

En primer lugar, la **media** presenta un comportamiento cada vez más cercano al valor original a medida que aumenta el tamaño de la muestra. Con 1.000 observaciones se obtiene una diferencia aproximada del 3,74 %, mientras que con 10.000 observaciones esta diferencia disminuye hasta aproximadamente el 0,45 %. Esto muestra que la muestra de mayor tamaño representa con mayor precisión el nivel promedio de las pérdidas.

La **varianza** presenta un comportamiento similar. Con la muestra de 1.000 observaciones existe una diferencia aproximada del 4,22 % frente a los datos originales, mientras que con 10.000 observaciones la diferencia disminuye hasta aproximadamente el 0,09 %. Este resultado indica que la muestra de mayor tamaño logra conservar de manera mucho más precisa la dispersión existente en la distribución original. En otras palabras, no solamente se aproxima mejor al promedio de las pérdidas, sino también a la forma en que estas varían alrededor de dicho promedio.

En cuanto a la **asimetría**, los resultados mantienen valores positivos tanto en los datos originales como en las dos muestras. Esto permite identificar una distribución con una marcada inclinación hacia la derecha, indicando que la mayoría de las observaciones se encuentran concentradas en niveles de pérdida relativamente menores, mientras que existe un grupo más reducido de observaciones con pérdidas considerablemente mayores. Esta característica resulta importante para el análisis del riesgo, ya que demuestra que el comportamiento de las pérdidas no puede comprenderse únicamente a partir de su valor promedio.

Además, la muestra de 10.000 observaciones presenta una asimetría más cercana a la distribución original que la muestra de 1.000. Esto demuestra nuevamente que un mayor tamaño de muestra permite representar mejor no solo el promedio y la dispersión de los datos, sino también la forma que presenta la distribución.

Por otra parte, la **curtosis** es la medida que presenta las mayores diferencias durante el proceso de muestreo. Con 1.000 observaciones la diferencia frente al valor original es aproximadamente del 9,34 %, mientras que con 10.000 disminuye hasta cerca del 2,94 %. Esto permite observar que las características asociadas con la concentración de los datos y la presencia de valores extremos son más sensibles al tamaño de la muestra que medidas como la media o la varianza.

Al comparar de manera general ambos tamaños de muestra, se observa una tendencia clara: **al aumentar el número de observaciones, las medidas estadísticas se aproximan a las características de la distribución original**. Esta mejora es especialmente evidente en la media y la varianza, aunque también se presenta en la asimetría y la curtosis.

Desde la perspectiva del problema analizado, este comportamiento es relevante porque la distribución presenta una alta variabilidad y una asimetría positiva. Esto significa que existen pérdidas que pueden alejarse considerablemente del comportamiento promedio. Por esta razón, para comprender adecuadamente el riesgo no resulta suficiente analizar únicamente la media; también es necesario estudiar la dispersión y la forma de la distribución.

En conclusión, **la muestra de 10.000 observaciones presenta una representación más precisa y estable de los datos originales que la muestra de 1.000 observaciones**. Los resultados muestran que aumentar el tamaño de la muestra reduce las diferencias en las medidas estadísticas analizadas y permite conservar con mayor precisión las características de la distribución original. Esto demuestra la importancia del tamaño de la muestra cuando se busca realizar una estimación confiable del comportamiento de las pérdidas y utilizar los resultados como apoyo para la toma de decisiones.
